# Home Credit：表内聚合（二）

## 5. `person_2`

`person_2` 是贷款申请下面的人员嵌套明细表。`case_id` 表示一笔贷款申请，`num_group1` 区分这笔申请关联的不同人员；这些人员可能是主申请人、共同申请人、联系人或亲属，并不一定都是贷款人。`num_group2` 继续区分同一人员下面的多条地址、就业、联系人或关联人员明细，但它只是结构序号，没有明确的业务类型或时间顺序。

因此不能直接按照 `num_group2` 选择第一条、最后一条或最新记录。本节先用代码确认三层结构和各类信息的分布，再按 `(case_id, num_group1)` 生成人员级特征，最后按 `case_id` 汇总成每笔贷款申请一行。人员级聚合完成后，再按 `case_id` 汇总总人数和明细数量，并保留前 4 位人员的聚合信息。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 表示人员层级；`num_group2` 表示该人员下的明细编号。
- **日期字段**：入职日期转换为申请时的就业年限。
- **低基数类别**：地址角色、联系人角色、经济状态和关联人员角色使用固定类别词表，统计每种类别的出现次数，并增加 `default` 和 `other` 类别。
- **高基数类别**：地区、邮编和雇主名称不全部展开，只统计有效数量和不同取值数量。
- **结构特征**：统计每个人的明细数、有效明细数，以及地址、就业、联系人和关联人员信息的数量。



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)

SPLIT = "train"
SAVE_RESULT = False
DEFAULT_CODE = "a55475b1"
MAX_PERSONS = 4

ROOTS = [
    Path("/kaggle/input/competitions/home-credit-credit-risk-model-stability"),
    Path("/kaggle/input/home-credit-credit-risk-model-stability"),
]
DATA_ROOT = next((path for path in ROOTS if path.exists()), None)
assert DATA_ROOT is not None, "找不到比赛数据。"

PARQUET_ROOT = DATA_ROOT / "parquet_files"
PERSON2_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_person_2.parquet"
BASE_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_base.parquet"
PERSON2_PERSON_OUTPUT = Path("/kaggle/working") / f"{SPLIT}_person_2_person_level.parquet"
PERSON2_CASE_OUTPUT = Path("/kaggle/working") / f"{SPLIT}_person_2_agg.parquet"

### 读取 `person_2`

这里只读取聚合需要的字段，并检查三列组合键是否唯一。

In [ ]:
PERSON2_DATE_COL = "empls_employedfrom_796D"
PERSON2_CATEGORY_COLS = [
    "addres_district_368M", "addres_role_871L", "addres_zip_823M",
    "conts_role_79M", "empls_economicalst_849M",
    "empls_employer_name_740M", "relatedpersons_role_762T",
]
PERSON2_LOW_CARD_VALUES = {
    "addres_role_871L": [
        "PERMANENT", "CONTACT", "TEMPORARY", "REGISTERED",
        "MIGRATED_REGISTRATION", "MIGRATED_LIVING", "MIGRATED_WORK", "MIGRATED_OTHER",
    ],
    "conts_role_79M": [
        "P38_92_157", "P177_137_98", "P7_147_157", "P125_105_50",
        "P115_147_77", "P125_14_176", "P58_79_51", "P124_137_181",
        "P206_38_166", "P42_134_91",
    ],
    "empls_economicalst_849M": [
        "P22_131_138", "P164_110_33", "P28_32_178", "P148_57_109",
        "P112_86_147", "P191_80_124", "P7_47_145", "P164_122_65", "P82_144_169",
    ],
    "relatedpersons_role_762T": [
        "OTHER_RELATIVE", "SIBLING", "FRIEND", "COLLEAGUE", "PARENT",
        "OTHER", "CHILD", "SPOUSE", "NEIGHBOR", "GRAND_PARENT",
    ],
}
PERSON2_COLS = [
    "case_id", "num_group1", "num_group2",
    *PERSON2_CATEGORY_COLS, PERSON2_DATE_COL,
]

person2 = pd.read_parquet(PERSON2_PATH, columns=PERSON2_COLS)
base_dates = pd.read_parquet(BASE_PATH, columns=["case_id", "date_decision"])
person2[PERSON2_DATE_COL] = pd.to_datetime(person2[PERSON2_DATE_COL], errors="coerce")
base_dates["date_decision"] = pd.to_datetime(base_dates["date_decision"], errors="coerce")

assert not person2.duplicated(["case_id", "num_group1", "num_group2"]).any()
print("person_2:", person2.shape)

### 聚合前结构确认

检查每笔申请的行数、关联人数，以及每位人员的明细数。

In [ ]:
rows_per_case = person2.groupby("case_id", sort=False).size()
rows_per_person = person2.groupby(["case_id", "num_group1"], sort=False).size()
persons_per_case = (
    person2[["case_id", "num_group1"]]
    .drop_duplicates()
    .groupby("case_id", sort=False)
    .size()
)
person2_structure = pd.DataFrame([{
    "n_rows": len(person2),
    "n_columns": person2.shape[1],
    "n_cases": person2["case_id"].nunique(),
    "composite_duplicate_rows": person2.duplicated(["case_id", "num_group1", "num_group2"]).sum(),
    "case_rate_le_1": rows_per_case.le(1).mean(),
    "case_rate_le_3": rows_per_case.le(3).mean(),
    "rows_per_case_max": rows_per_case.max(),
    "persons_per_case_max": persons_per_case.max(),
    "rows_per_person_p50": rows_per_person.median(),
    "rows_per_person_p99": rows_per_person.quantile(0.99),
    "rows_per_person_max": rows_per_person.max(),
}])
display(person2_structure)

### 信息组合检查

检查每行是否包含地址、就业、联系人或关联人员信息，并计算空明细比例。

In [ ]:
def has_nondefault(df, columns):
    values = df[columns]
    return (values.notna() & values.ne(DEFAULT_CODE)).any(axis=1)

has_address = has_nondefault(person2, [
    "addres_district_368M", "addres_role_871L", "addres_zip_823M",
])
has_employment = (
    has_nondefault(person2, ["empls_economicalst_849M", "empls_employer_name_740M"])
    | person2[PERSON2_DATE_COL].notna()
)
has_contact = has_nondefault(person2, ["conts_role_79M"])
has_related_person = has_nondefault(person2, ["relatedpersons_role_762T"])
has_business_value = has_address | has_employment | has_contact | has_related_person

field_pattern = (
    pd.DataFrame({
        "has_address": has_address,
        "has_employment": has_employment,
        "has_contact": has_contact,
        "has_related_person": has_related_person,
    })
    .value_counts(sort=True)
    .rename("n_rows")
    .reset_index()
)
print("空明细比例：", round((~has_business_value).mean(), 6))
display(field_pattern)

### 第一层：聚合到人员

按 `(case_id, num_group1)` 整理成每位人员一行：

- `num_group2` 只用于统计明细数，之后删除；
- 低基数类别统计各类别次数，高基数编码统计有效数和不同值数；
- 就业日期转换为相对申请日的就业期限。

In [ ]:
def prepare_person2_detail(df, base):
    x = df.merge(base, on="case_id", how="left", validate="many_to_one").copy()
    x["_employment_tenure_days"] = (x["date_decision"] - x[PERSON2_DATE_COL]).dt.days

    clean = {}
    for col in PERSON2_CATEGORY_COLS:
        clean[col] = f"_clean_{col}"
        x[clean[col]] = x[col].replace(DEFAULT_CODE, pd.NA)

    category_count_aggs = {}
    for col, values in PERSON2_LOW_CARD_VALUES.items():
        short_name = col.rsplit("_", 1)[0]
        for index, value in enumerate(values):
            flag_col = f"_category_{short_name}_{index}"
            x[flag_col] = x[col].eq(value).fillna(False).astype("int8")
            feature_name = f"person2_{short_name}_{value.lower()}_count"
            category_count_aggs[feature_name] = (flag_col, "sum")

        default_col = f"_category_{short_name}_default"
        other_col = f"_category_{short_name}_other"
        x[default_col] = (x[col].isna() | x[col].eq(DEFAULT_CODE).fillna(False)).astype("int8")
        x[other_col] = (
            x[col].notna()
            & ~x[col].eq(DEFAULT_CODE).fillna(False)
            & ~x[col].isin(values)
        ).astype("int8")
        category_count_aggs[f"person2_{short_name}_default_count"] = (default_col, "sum")
        category_count_aggs[f"person2_{short_name}_other_count"] = (other_col, "sum")

    x["_has_address"] = x[[
        clean["addres_district_368M"], clean["addres_role_871L"],
        clean["addres_zip_823M"],
    ]].notna().any(axis=1)
    x["_has_employment"] = x[[
        clean["empls_economicalst_849M"], clean["empls_employer_name_740M"],
        PERSON2_DATE_COL,
    ]].notna().any(axis=1)
    x["_has_contact"] = x[clean["conts_role_79M"]].notna()
    x["_has_related_person"] = x[clean["relatedpersons_role_762T"]].notna()
    family_flags = ["_has_address", "_has_employment", "_has_contact", "_has_related_person"]
    x["_has_meaningful_detail"] = x[family_flags].any(axis=1)

    return x, clean, category_count_aggs

### 生成人员级结果

将准备好的明细按 `(case_id, num_group1)` 汇总，并保留有效、空白及各信息模块的数量。

In [ ]:
def aggregate_person2_to_person(df, base):
    x, clean, category_count_aggs = prepare_person2_detail(df, base)
    g = x.groupby(["case_id", "num_group1"], sort=False)
    features = g.agg(
        person2_detail_count=("num_group2", "size"),
        person2_meaningful_detail_count=("_has_meaningful_detail", "sum"),
        person2_address_count=("_has_address", "sum"),
        person2_employment_count=("_has_employment", "sum"),
        person2_contact_count=("_has_contact", "sum"),
        person2_related_person_count=("_has_related_person", "sum"),
        person2_address_role_nunique=(clean["addres_role_871L"], "nunique"),
        person2_address_district_nunique=(clean["addres_district_368M"], "nunique"),
        person2_address_zip_nunique=(clean["addres_zip_823M"], "nunique"),
        person2_contact_role_nunique=(clean["conts_role_79M"], "nunique"),
        person2_economic_status_nunique=(clean["empls_economicalst_849M"], "nunique"),
        person2_known_employer_count=(clean["empls_employer_name_740M"], "count"),
        person2_employer_nunique=(clean["empls_employer_name_740M"], "nunique"),
        person2_related_role_nunique=(clean["relatedpersons_role_762T"], "nunique"),
        person2_employment_date_count=(PERSON2_DATE_COL, "count"),
        person2_employment_tenure_days_min=("_employment_tenure_days", "min"),
        person2_employment_tenure_days_mean=("_employment_tenure_days", "mean"),
        person2_employment_tenure_days_max=("_employment_tenure_days", "max"),
        **category_count_aggs,
    )
    features["person2_empty_detail_count"] = (
        features["person2_detail_count"] - features["person2_meaningful_detail_count"]
    )
    features["person2_meaningful_detail_rate"] = (
        features["person2_meaningful_detail_count"] / features["person2_detail_count"]
    )
    return features.reset_index()

### 第二层：聚合到贷款申请

把最多 4 位人员的特征分别展开为独立列，同时生成申请级人数和明细汇总。人员位置只是结构编号，不代表重要程度或角色。

In [ ]:
def aggregate_person2_to_case(person_df, max_persons=4):
    g = person_df.groupby("case_id", sort=False)
    features = g.agg(
        person2_person_count=("num_group1", "size"),
        person2_detail_count_sum=("person2_detail_count", "sum"),
        person2_detail_count_mean=("person2_detail_count", "mean"),
        person2_detail_count_max=("person2_detail_count", "max"),
        person2_meaningful_detail_count_sum=("person2_meaningful_detail_count", "sum"),
        person2_empty_detail_count_sum=("person2_empty_detail_count", "sum"),
        person2_address_count_sum=("person2_address_count", "sum"),
        person2_employment_count_sum=("person2_employment_count", "sum"),
        person2_contact_count_sum=("person2_contact_count", "sum"),
        person2_related_person_count_sum=("person2_related_person_count", "sum"),
        person2_known_employer_count_sum=("person2_known_employer_count", "sum"),
        person2_employment_date_count_sum=("person2_employment_date_count", "sum"),
    )

    person_cols = [col for col in person_df.columns if col not in {"case_id", "num_group1"}]
    ranked = person_df.sort_values(["case_id", "num_group1"]).copy()
    ranked["_rank"] = ranked.groupby("case_id").cumcount() + 1
    for rank in range(1, max_persons + 1):
        part = ranked.loc[ranked["_rank"] == rank, ["case_id", *person_cols]].set_index("case_id")
        part = part.rename(columns={col: f"person2_person{rank}_{col.removeprefix('person2_')}" for col in person_cols})
        features = features.join(part)

    features["person2_meaningful_detail_rate"] = (
        features["person2_meaningful_detail_count_sum"] / features["person2_detail_count_sum"]
    )
    features["person2_has_meaningful_detail"] = (
        features["person2_meaningful_detail_count_sum"] > 0
    ).astype("int8")
    features["person2_has_record"] = 1
    features["person2_detail_count_log1p"] = np.log1p(features["person2_detail_count_sum"])
    return features.reset_index()

### 执行聚合

先按 `(case_id, num_group1)` 生成可与 `person_1` 对齐的人员级结果，再按 `case_id` 生成每笔贷款申请一行的申请级结果。

In [ ]:
person2_person_features = aggregate_person2_to_person(person2, base_dates)
person2_features = aggregate_person2_to_case(person2_person_features, MAX_PERSONS)

assert not person2_person_features.duplicated(["case_id", "num_group1"]).any()
assert person2_features["case_id"].is_unique
assert len(person2_features) == person2["case_id"].nunique()

print("人员级结果：", person2_person_features.shape)
print("申请级结果：", person2_features.shape)
display(person2_features.head())

if SAVE_RESULT:
    person2_person_features.to_parquet(PERSON2_PERSON_OUTPUT, index=False)
    person2_features.to_parquet(PERSON2_CASE_OUTPUT, index=False)
    print("已保存：", PERSON2_PERSON_OUTPUT)
    print("已保存：", PERSON2_CASE_OUTPUT)

## 6. `tax_registry_a_1`

`tax_registry_a_1` 是重复税务明细表。一笔贷款申请可以对应多条扣款金额记录，`(case_id, num_group1)` 是原始行粒度；`num_group1` 只是明细序号，并不代表雇主或时间顺序。

全表中每个 `case_id` 只有一个登记日期，因此 `recorddate_4527225D` 是整批税务记录的登记日期，而不是多期历史时间轴。约 29.60% 的申请涉及多个雇主，所以先按雇主汇总，再聚合到贷款申请。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是税务记录编号。
- **数值字段**：扣款金额计算 `count / sum / mean / median / min / max / std`，并增加零值数量、金额范围及变异系数。
- **日期字段**：记录日期用于计算距申请日天数、不同记录日期数量，以及是否存在申请日后的记录。
- **高基数类别**：雇主名称不全部展开，而是统计雇主数量、各雇主金额、最大雇主金额、主要雇主占比和雇主集中度，并保留主要雇主代码。

In [ ]:
TAX_A_AMOUNT = "amount_4527230A"
TAX_A_EMPLOYER = "name_4527232M"
TAX_A_DATE = "recorddate_4527225D"
TAX_A_COLS = ["case_id", "num_group1", TAX_A_AMOUNT, TAX_A_EMPLOYER, TAX_A_DATE]

TAX_A_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_tax_registry_a_1.parquet"
TAX_A_OUTPUT = Path("/kaggle/working") / f"{SPLIT}_tax_registry_a_1_agg.parquet"

tax_a = pd.read_parquet(TAX_A_PATH, columns=TAX_A_COLS)
tax_a[TAX_A_DATE] = pd.to_datetime(tax_a[TAX_A_DATE], errors="coerce")
tax_a_base = pd.read_parquet(BASE_PATH, columns=["case_id", "date_decision"])
tax_a_base["date_decision"] = pd.to_datetime(tax_a_base["date_decision"], errors="coerce")

assert not tax_a.duplicated(["case_id", "num_group1"]).any()
assert tax_a_base["case_id"].is_unique
print("tax_registry_a_1:", tax_a.shape)

### 时间与雇主结构确认

这一步检查同一笔申请是否存在多个登记日期和多个雇主。日期数量决定能否使用历史时间窗口，雇主数量决定是否需要先做雇主级聚合。

In [ ]:
tax_case_check = tax_a.groupby("case_id", sort=False).agg(
    n_rows=("num_group1", "size"),
    n_dates=(TAX_A_DATE, "nunique"),
    n_employers=(TAX_A_EMPLOYER, "nunique"),
)

tax_structure = pd.DataFrame([{
    "n_cases": len(tax_case_check),
    "multi_date_case_rate": tax_case_check["n_dates"].gt(1).mean(),
    "dates_p50": tax_case_check["n_dates"].median(),
    "dates_p90": tax_case_check["n_dates"].quantile(0.90),
    "dates_max": tax_case_check["n_dates"].max(),
    "multi_employer_case_rate": tax_case_check["n_employers"].gt(1).mean(),
    "employers_p50": tax_case_check["n_employers"].median(),
    "employers_max": tax_case_check["n_employers"].max(),
}])
display(tax_structure)

### 金额与雇主字段质量检查

这一步检查金额的缺失、零值和负数，并检查雇主名称中的空值和默认编码。它决定聚合前是否需要删除、替换或额外标记异常值。

In [ ]:
amount = tax_a[TAX_A_AMOUNT]
employer = tax_a[TAX_A_EMPLOYER]
valid_amount_count = amount.notna().sum()

tax_field_check = pd.DataFrame([{
    "n_rows": len(tax_a),
    "amount_missing_count": amount.isna().sum(),
    "amount_zero_count": amount.eq(0).sum(),
    "amount_negative_count": amount.lt(0).sum(),
    "amount_positive_count": amount.gt(0).sum(),
    "amount_min": amount.min(),
    "amount_median": amount.median(),
    "amount_mean": amount.mean(),
    "amount_max": amount.max(),
    "employer_null_count": employer.isna().sum(),
    "employer_empty_count": employer.eq("").sum(),
    "employer_default_count": employer.eq(DEFAULT_CODE).sum(),
    "employer_raw_nunique": employer.nunique(dropna=True),
    "amount_missing_rate": amount.isna().mean(),
    "amount_zero_rate_among_valid": amount.eq(0).sum() / max(valid_amount_count, 1),
    "employer_missing_or_default_rate": (
        employer.isna() | employer.eq("") | employer.eq(DEFAULT_CODE)
    ).mean(),
}])
display(tax_field_check)

### 高频雇主编码检查

这一步查看出现次数最多的雇主编码，用于发现隐藏的默认占位值或近常数编码。高频本身不是删除理由；没有业务证据时仍按正常雇主编码处理。

In [ ]:
employer_top_values = (
    tax_a[TAX_A_EMPLOYER]
    .value_counts(dropna=False)
    .head(15)
    .rename_axis(TAX_A_EMPLOYER)
    .reset_index(name="count")
)
employer_top_values["rate"] = employer_top_values["count"] / len(tax_a)
display(employer_top_values)

### 聚合思路

先按 `(case_id, name_4527232M)` 汇总同一雇主的多条税务明细，再按 `case_id` 汇总到贷款申请。这样既保留整体扣款金额分布，也保留雇主数量、主要雇主金额和雇主集中度。

`recorddate_4527225D` 在每笔申请内只有一个值，因此只计算它距离 `date_decision` 的天数，不做月度窗口或趋势。`num_group1` 只用于确认原始粒度和统计明细数，聚合后删除。高基数雇主编码不做 One-Hot；仅保留扣款金额最大的雇主编码，供后续建模阶段决定是否进行频率编码或按类别处理。

In [ ]:
def aggregate_tax_registry_a(df, base):
    x = df.copy()

    employer_level = (
        x.groupby(["case_id", TAX_A_EMPLOYER], sort=False, dropna=False)
        .agg(
            taxa_employer_record_count=(TAX_A_AMOUNT, "size"),
            taxa_employer_amount_sum=(TAX_A_AMOUNT, "sum"),
            taxa_employer_amount_mean=(TAX_A_AMOUNT, "mean"),
            taxa_employer_amount_min=(TAX_A_AMOUNT, "min"),
            taxa_employer_amount_max=(TAX_A_AMOUNT, "max"),
            taxa_employer_amount_std=(TAX_A_AMOUNT, "std"),
        )
        .reset_index()
    )

    employer_level["_case_amount_total"] = (
        employer_level.groupby("case_id")["taxa_employer_amount_sum"].transform("sum")
    )
    employer_level["taxa_employer_amount_share"] = np.where(
        employer_level["_case_amount_total"].ne(0),
        employer_level["taxa_employer_amount_sum"] / employer_level["_case_amount_total"],
        0.0,
    )
    employer_level["_share_squared"] = employer_level["taxa_employer_amount_share"].pow(2)

    employer_summary = (
        employer_level.groupby("case_id", sort=False)
        .agg(
            taxa_employer_count=(TAX_A_EMPLOYER, "size"),
            taxa_employer_amount_mean=("taxa_employer_amount_sum", "mean"),
            taxa_employer_amount_std=("taxa_employer_amount_sum", "std"),
            taxa_largest_employer_amount=("taxa_employer_amount_sum", "max"),
            taxa_largest_employer_record_count=("taxa_employer_record_count", "max"),
            taxa_top_employer_share=("taxa_employer_amount_share", "max"),
            taxa_employer_amount_hhi=("_share_squared", "sum"),
        )
        .reset_index()
    )

    top_employer = (
        employer_level.sort_values(
            ["case_id", "taxa_employer_amount_sum", "taxa_employer_record_count"],
            ascending=[True, False, False],
        )
        .drop_duplicates("case_id")
        [["case_id", TAX_A_EMPLOYER]]
        .rename(columns={TAX_A_EMPLOYER: "taxa_top_employer_code"})
    )

    features = (
        x.groupby("case_id", sort=False)
        .agg(
            taxa_record_count=("num_group1", "size"),
            taxa_amount_sum=(TAX_A_AMOUNT, "sum"),
            taxa_amount_mean=(TAX_A_AMOUNT, "mean"),
            taxa_amount_median=(TAX_A_AMOUNT, "median"),
            taxa_amount_min=(TAX_A_AMOUNT, "min"),
            taxa_amount_max=(TAX_A_AMOUNT, "max"),
            taxa_amount_std=(TAX_A_AMOUNT, "std"),
            taxa_amount_zero_count=(TAX_A_AMOUNT, lambda s: s.eq(0).sum()),
            taxa_record_date=(TAX_A_DATE, "first"),
            taxa_record_date_nunique=(TAX_A_DATE, "nunique"),
        )
        .reset_index()
        .merge(employer_summary, on="case_id", how="left", validate="one_to_one")
        .merge(top_employer, on="case_id", how="left", validate="one_to_one")
        .merge(base, on="case_id", how="left", validate="one_to_one")
    )

    features["taxa_record_recency_days"] = (
        features["date_decision"] - features["taxa_record_date"]
    ).dt.days
    features["taxa_record_after_decision"] = features["taxa_record_recency_days"].lt(0).astype("int8")
    features["taxa_amount_range"] = features["taxa_amount_max"] - features["taxa_amount_min"]
    features["taxa_amount_cv"] = (
        features["taxa_amount_std"] / features["taxa_amount_mean"].replace(0, np.nan)
    )
    features["taxa_has_record"] = 1
    features["taxa_record_count_log1p"] = np.log1p(features["taxa_record_count"])
    features["taxa_amount_sum_log1p"] = np.log1p(features["taxa_amount_sum"].clip(lower=0))
    features = features.drop(columns=["taxa_record_date", "date_decision"])

    return employer_level.drop(columns=["_case_amount_total", "_share_squared"]), features


### 执行聚合

生成雇主级中间结果和每笔贷款申请一行的最终结果。`SAVE_RESULT=False` 时只检查结果，不写入文件。

In [ ]:
taxa_employer_features, taxa_features = aggregate_tax_registry_a(tax_a, tax_a_base)

assert taxa_features["case_id"].is_unique
assert len(taxa_features) == tax_a["case_id"].nunique()

print("雇主级结果：", taxa_employer_features.shape)
print("申请级结果：", taxa_features.shape)
display(taxa_features.head())

if SAVE_RESULT:
    taxa_features.to_parquet(TAX_A_OUTPUT, index=False)
    print("已保存：", TAX_A_OUTPUT)

## 7. `tax_registry_b_1`

`tax_registry_b_1` 是一张税务扣款历史长表，一行代表一条带有日期的税务扣款记录，`case_id + num_group1` 唯一定位一条记录。最终按 `case_id` 聚合为每个客户一行。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是税务记录编号。
- **数值字段**：扣款金额计算 `count / sum / mean / median / min / max / std`，并比较最早与最新日期的每日金额变化。
- **日期字段**：计算有效日期数、日期跨度、最新记录距申请日天数，以及相邻记录的时间间隔。
- **高基数类别**：雇主名称不全部展开，而是统计雇主数量、最大雇主金额和主要雇主金额占比。
- **重复记录**：根据金额、日期和雇主识别业务重复，计算重复记录数量及比例。

另外统计申请日前最近 30、90 和 180 天的记录数、金额总和及活跃日期数，以保留近期税务活动。

In [ ]:
TAX_B_AMOUNT = "amount_4917619A"
TAX_B_DATE = "deductiondate_4917603D"
TAX_B_EMPLOYER = "name_4917606M"
TAX_B_COLS = ["case_id", "num_group1", TAX_B_AMOUNT, TAX_B_DATE, TAX_B_EMPLOYER]

TAX_B_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_tax_registry_b_1.parquet"
TAX_B_OUTPUT = Path("/kaggle/working") / f"{SPLIT}_tax_registry_b_1_agg.parquet"

tax_b = pd.read_parquet(TAX_B_PATH, columns=TAX_B_COLS)
tax_b[TAX_B_DATE] = pd.to_datetime(tax_b[TAX_B_DATE], errors="coerce")
tax_b_base = pd.read_parquet(BASE_PATH, columns=["case_id", "date_decision"])
tax_b_base["date_decision"] = pd.to_datetime(tax_b_base["date_decision"], errors="coerce")

assert not tax_b.duplicated(["case_id", "num_group1"]).any()
assert tax_b_base["case_id"].is_unique
print("tax_registry_b_1:", tax_b.shape)

### 检查时间和重复记录

确认每笔申请有多少个扣款日期，并统计业务字段完全相同的记录比例。

In [ ]:
taxb_case_check = tax_b.groupby("case_id", sort=False).agg(
    n_rows=("num_group1", "size"),
    n_dates=(TAX_B_DATE, "nunique"),
    date_min=(TAX_B_DATE, "min"),
    date_max=(TAX_B_DATE, "max"),
    n_employers=(TAX_B_EMPLOYER, "nunique"),
)
taxb_case_check["span_days"] = (taxb_case_check["date_max"] - taxb_case_check["date_min"]).dt.days

business_cols_b = ["case_id", TAX_B_AMOUNT, TAX_B_DATE, TAX_B_EMPLOYER]
taxb_unique_business = tax_b.drop_duplicates(business_cols_b)

taxb_check = pd.DataFrame([{
    "n_cases": len(taxb_case_check),
    "multi_date_case_rate": taxb_case_check["n_dates"].gt(1).mean(),
    "dates_p50": taxb_case_check["n_dates"].median(),
    "dates_p90": taxb_case_check["n_dates"].quantile(0.90),
    "span_days_p50": taxb_case_check["span_days"].median(),
    "multi_employer_case_rate": taxb_case_check["n_employers"].gt(1).mean(),
    "extra_duplicate_row_rate": 1 - len(taxb_unique_business) / len(tax_b),
}])
display(taxb_check)

### 整理日级和雇主级数据

先统计每天和每个雇主的扣款金额，保留时间与雇主结构。

In [ ]:
def prepare_tax_registry_b(df, base):
    x = df.merge(base, on="case_id", how="left", validate="many_to_one").copy()
    x["_days_before_decision"] = (x["date_decision"] - x[TAX_B_DATE]).dt.days
    unique_business = x.drop_duplicates(["case_id", TAX_B_AMOUNT, TAX_B_DATE, TAX_B_EMPLOYER])

    daily = (
        x.groupby(["case_id", TAX_B_DATE], sort=False, dropna=False)
        .agg(
            taxb_daily_record_count=(TAX_B_AMOUNT, "size"),
            taxb_daily_amount_sum=(TAX_B_AMOUNT, "sum"),
        )
        .reset_index()
        .sort_values(["case_id", TAX_B_DATE])
    )
    daily["_gap_days"] = daily.groupby("case_id")[TAX_B_DATE].diff().dt.days

    employer = (
        x.groupby(["case_id", TAX_B_EMPLOYER], sort=False, dropna=False)
        .agg(taxb_employer_amount_sum=(TAX_B_AMOUNT, "sum"))
        .reset_index()
    )
    employer["_case_amount_total"] = employer.groupby("case_id")["taxb_employer_amount_sum"].transform("sum")
    employer["_amount_share"] = np.where(
        employer["_case_amount_total"].ne(0),
        employer["taxb_employer_amount_sum"] / employer["_case_amount_total"],
        0.0,
    )
    return x, unique_business, daily, employer

### 聚合到贷款申请

保留金额、日期跨度、时间间隔、雇主数量、重复比例，以及最近 30/90/180 天的扣款特征。时间窗口使用实际天数，不假设月份连续。

In [ ]:
def aggregate_tax_registry_b(df, base):
    x, unique_business, daily, employer = prepare_tax_registry_b(df, base)

    features = x.groupby("case_id", sort=False).agg(
        taxb_record_count=("num_group1", "size"),
        taxb_amount_sum=(TAX_B_AMOUNT, "sum"),
        taxb_amount_mean=(TAX_B_AMOUNT, "mean"),
        taxb_amount_median=(TAX_B_AMOUNT, "median"),
        taxb_amount_min=(TAX_B_AMOUNT, "min"),
        taxb_amount_max=(TAX_B_AMOUNT, "max"),
        taxb_amount_std=(TAX_B_AMOUNT, "std"),
        taxb_date_count=(TAX_B_DATE, "nunique"),
        taxb_date_min=(TAX_B_DATE, "min"),
        taxb_date_max=(TAX_B_DATE, "max"),
        taxb_employer_count=(TAX_B_EMPLOYER, "nunique"),
        taxb_after_decision_count=("_days_before_decision", lambda s: s.lt(0).sum()),
    ).reset_index()

    unique_summary = unique_business.groupby("case_id", sort=False).agg(
        taxb_unique_record_count=("num_group1", "size"),
        taxb_unique_amount_sum=(TAX_B_AMOUNT, "sum"),
    ).reset_index()
    daily_summary = daily.groupby("case_id", sort=False).agg(
        taxb_gap_days_mean=("_gap_days", "mean"),
        taxb_gap_days_std=("_gap_days", "std"),
        taxb_gap_days_max=("_gap_days", "max"),
    ).reset_index()
    earliest = daily.drop_duplicates("case_id")[["case_id", "taxb_daily_amount_sum"]].rename(
        columns={"taxb_daily_amount_sum": "taxb_earliest_daily_amount"}
    )
    latest = daily.drop_duplicates("case_id", keep="last")[["case_id", "taxb_daily_amount_sum"]].rename(
        columns={"taxb_daily_amount_sum": "taxb_latest_daily_amount"}
    )
    employer_summary = employer.groupby("case_id", sort=False).agg(
        taxb_largest_employer_amount=("taxb_employer_amount_sum", "max"),
        taxb_top_employer_share=("_amount_share", "max"),
    ).reset_index()

    for part in [unique_summary, daily_summary, earliest, latest, employer_summary]:
        features = features.merge(part, on="case_id", how="left", validate="one_to_one")

    for days in [30, 90, 180]:
        recent = x[x["_days_before_decision"].between(0, days)]
        window = recent.groupby("case_id", sort=False).agg(
            **{
                f"taxb_record_count_{days}d": ("num_group1", "size"),
                f"taxb_amount_sum_{days}d": (TAX_B_AMOUNT, "sum"),
                f"taxb_active_dates_{days}d": (TAX_B_DATE, "nunique"),
            }
        ).reset_index()
        features = features.merge(window, on="case_id", how="left", validate="one_to_one")
        window_cols = [
            f"taxb_record_count_{days}d", f"taxb_amount_sum_{days}d",
            f"taxb_active_dates_{days}d",
        ]
        features[window_cols] = features[window_cols].fillna(0)

    features = features.merge(base, on="case_id", how="left", validate="one_to_one")
    features["taxb_duplicate_record_count"] = features["taxb_record_count"] - features["taxb_unique_record_count"]
    features["taxb_duplicate_record_rate"] = features["taxb_duplicate_record_count"] / features["taxb_record_count"]
    features["taxb_date_span_days"] = (features["taxb_date_max"] - features["taxb_date_min"]).dt.days
    features["taxb_latest_recency_days"] = (features["date_decision"] - features["taxb_date_max"]).dt.days
    features["taxb_daily_amount_change"] = features["taxb_latest_daily_amount"] - features["taxb_earliest_daily_amount"]
    features["taxb_has_record"] = 1
    features["taxb_record_count_log1p"] = np.log1p(features["taxb_record_count"])
    features = features.drop(columns=["taxb_date_min", "taxb_date_max", "date_decision"])
    return features

### 执行聚合

生成每笔贷款申请一行的结果，并检查 `case_id` 是否唯一。

In [ ]:
taxb_features = aggregate_tax_registry_b(tax_b, tax_b_base)

assert taxb_features["case_id"].is_unique
assert len(taxb_features) == tax_b["case_id"].nunique()

print("申请级结果：", taxb_features.shape)
display(taxb_features.head())

if SAVE_RESULT:
    taxb_features.to_parquet(TAX_B_OUTPUT, index=False)
    print("已保存：", TAX_B_OUTPUT)

## 8. `tax_registry_c_1`

`tax_registry_c_1` 是一张税务付款历史长表，一行代表一条带有处理日期的税务付款记录，`case_id + num_group1` 唯一定位一条记录。最终按 `case_id` 聚合为每个客户一行。

### 字段及处理方式

- **键字段**：`case_id` 用于分组和连接；`num_group1` 是税务记录编号。
- **数值字段**：付款金额计算 `count / sum / mean / median / min / max / std`，并比较最早与最新日期的每日金额变化。
- **日期字段**：计算有效日期数、日期跨度、最新记录距申请日天数，以及相邻记录的时间间隔。
- **高基数类别**：雇主名称不全部展开，而是统计雇主数量、最大雇主金额和主要雇主金额占比。
- **重复记录**：根据金额、日期和雇主识别业务重复，计算重复记录数量及比例。

另外统计申请日前最近 30、90 和 180 天的记录数、金额总和及活跃日期数，以保留近期税务活动。

与前两张表的区别：

- `tax_registry_a_1`：每笔申请只有一个登记日期，更像一次登记快照，不做时间窗口。
- `tax_registry_b_1`：记录实际扣款日期，是多期扣款历史。
- `tax_registry_c_1`：记录扣款的处理日期，同样是多期历史，但完全相同的记录更多。

三张表含义不同，因此分别聚合后再按 `case_id` 合并，不直接拼接原始记录。


### 第一步：读取数据

读取金额、处理日期、雇主和明细序号，并把处理日期转换为日期类型。


In [ ]:
TAX_C_AMOUNT = "pmtamount_36A"
TAX_C_DATE = "processingdate_168D"
TAX_C_EMPLOYER = "employername_160M"
TAX_C_COLS = ["case_id", "num_group1", TAX_C_AMOUNT, TAX_C_DATE, TAX_C_EMPLOYER]

TAX_C_PATH = PARQUET_ROOT / SPLIT / f"{SPLIT}_tax_registry_c_1.parquet"
TAX_C_OUTPUT = Path("/kaggle/working") / f"{SPLIT}_tax_registry_c_1_agg.parquet"

tax_c = pd.read_parquet(TAX_C_PATH, columns=TAX_C_COLS)
tax_c[TAX_C_DATE] = pd.to_datetime(tax_c[TAX_C_DATE], errors="coerce")
tax_c_base = pd.read_parquet(BASE_PATH, columns=["case_id", "date_decision"])
tax_c_base["date_decision"] = pd.to_datetime(tax_c_base["date_decision"], errors="coerce")

assert not tax_c.duplicated(["case_id", "num_group1"]).any()
assert tax_c_base["case_id"].is_unique
print("tax_registry_c_1:", tax_c.shape)


### 第二步：检查时间与雇主结构

检查每笔申请有多少个处理日期、记录覆盖多少天，以及是否关联多个雇主。这一步用于确认该表是否适合使用时间窗口和雇主统计。


In [ ]:
taxc_case_check = tax_c.groupby("case_id", sort=False).agg(
    n_rows=("num_group1", "size"),
    n_dates=(TAX_C_DATE, "nunique"),
    date_min=(TAX_C_DATE, "min"),
    date_max=(TAX_C_DATE, "max"),
    n_employers=(TAX_C_EMPLOYER, "nunique"),
)
taxc_case_check["span_days"] = (
    taxc_case_check["date_max"] - taxc_case_check["date_min"]
).dt.days

taxc_time_check = pd.DataFrame([{
    "n_cases": len(taxc_case_check),
    "multi_date_case_rate": taxc_case_check["n_dates"].gt(1).mean(),
    "dates_p50": taxc_case_check["n_dates"].median(),
    "dates_p90": taxc_case_check["n_dates"].quantile(0.90),
    "dates_max": taxc_case_check["n_dates"].max(),
    "span_days_p50": taxc_case_check["span_days"].median(),
    "span_days_p90": taxc_case_check["span_days"].quantile(0.90),
    "multi_employer_case_rate": taxc_case_check["n_employers"].gt(1).mean(),
}])
display(taxc_time_check)


### 第三步：检查完全相同的业务记录

忽略 `num_group1`，检查同一申请、日期、雇主和金额完全相同的记录。由于无法确认它们是真实的多笔相同扣款还是重复数据，这里不直接删除，而是为后续同时保留原始统计和业务去重统计。


In [ ]:
business_cols_c = ["case_id", TAX_C_AMOUNT, TAX_C_DATE, TAX_C_EMPLOYER]
taxc_group_sizes = tax_c.groupby(business_cols_c, dropna=False).size()
taxc_unique_business = tax_c.drop_duplicates(business_cols_c)

taxc_duplicate_check = pd.DataFrame([{
    "n_business_groups": len(taxc_group_sizes),
    "duplicate_group_rate": taxc_group_sizes.gt(1).mean(),
    "extra_duplicate_row_rate": 1 - len(taxc_unique_business) / len(tax_c),
    "max_identical_rows": taxc_group_sizes.max(),
}])
display(taxc_duplicate_check)


### 第四步：整理日级和雇主级数据

先按处理日期汇总每日金额，再按雇主汇总金额。同时生成业务去重版本，为申请级聚合准备中间数据。


In [ ]:
def prepare_tax_registry_c(df, base):
    x = df.merge(base, on="case_id", how="left", validate="many_to_one").copy()
    x["_days_before_decision"] = (x["date_decision"] - x[TAX_C_DATE]).dt.days
    unique_business = x.drop_duplicates(
        ["case_id", TAX_C_AMOUNT, TAX_C_DATE, TAX_C_EMPLOYER]
    )

    daily = (
        x.groupby(["case_id", TAX_C_DATE], sort=False, dropna=False)
        .agg(
            taxc_daily_record_count=(TAX_C_AMOUNT, "size"),
            taxc_daily_amount_sum=(TAX_C_AMOUNT, "sum"),
        )
        .reset_index()
        .sort_values(["case_id", TAX_C_DATE])
    )
    daily["_gap_days"] = daily.groupby("case_id")[TAX_C_DATE].diff().dt.days

    employer = (
        x.groupby(["case_id", TAX_C_EMPLOYER], sort=False, dropna=False)
        .agg(taxc_employer_amount_sum=(TAX_C_AMOUNT, "sum"))
        .reset_index()
    )
    employer["_case_amount_total"] = employer.groupby("case_id")[
        "taxc_employer_amount_sum"
    ].transform("sum")
    employer["_amount_share"] = np.where(
        employer["_case_amount_total"].ne(0),
        employer["taxc_employer_amount_sum"] / employer["_case_amount_total"],
        0.0,
    )
    return x, unique_business, daily, employer


### 第五步：聚合到贷款申请

最终生成每笔申请一行，保留金额分布、日期跨度、处理间隔、雇主结构和最近 30/90/180 天特征。原始记录与业务去重记录同时保留，用它们的差异描述重复程度。


In [ ]:
def aggregate_tax_registry_c(df, base):
    x, unique_business, daily, employer = prepare_tax_registry_c(df, base)

    features = x.groupby("case_id", sort=False).agg(
        taxc_record_count=("num_group1", "size"),
        taxc_amount_sum=(TAX_C_AMOUNT, "sum"),
        taxc_amount_mean=(TAX_C_AMOUNT, "mean"),
        taxc_amount_median=(TAX_C_AMOUNT, "median"),
        taxc_amount_min=(TAX_C_AMOUNT, "min"),
        taxc_amount_max=(TAX_C_AMOUNT, "max"),
        taxc_amount_std=(TAX_C_AMOUNT, "std"),
        taxc_date_count=(TAX_C_DATE, "nunique"),
        taxc_date_min=(TAX_C_DATE, "min"),
        taxc_date_max=(TAX_C_DATE, "max"),
        taxc_employer_count=(TAX_C_EMPLOYER, "nunique"),
        taxc_after_decision_count=("_days_before_decision", lambda s: s.lt(0).sum()),
    ).reset_index()

    unique_summary = unique_business.groupby("case_id", sort=False).agg(
        taxc_unique_record_count=("num_group1", "size"),
        taxc_unique_amount_sum=(TAX_C_AMOUNT, "sum"),
    ).reset_index()
    daily_summary = daily.groupby("case_id", sort=False).agg(
        taxc_gap_days_mean=("_gap_days", "mean"),
        taxc_gap_days_std=("_gap_days", "std"),
        taxc_gap_days_max=("_gap_days", "max"),
    ).reset_index()
    earliest = daily.drop_duplicates("case_id")[["case_id", "taxc_daily_amount_sum"]].rename(
        columns={"taxc_daily_amount_sum": "taxc_earliest_daily_amount"}
    )
    latest = daily.drop_duplicates("case_id", keep="last")[[
        "case_id", "taxc_daily_amount_sum"
    ]].rename(columns={"taxc_daily_amount_sum": "taxc_latest_daily_amount"})
    employer_summary = employer.groupby("case_id", sort=False).agg(
        taxc_largest_employer_amount=("taxc_employer_amount_sum", "max"),
        taxc_top_employer_share=("_amount_share", "max"),
    ).reset_index()

    for part in [unique_summary, daily_summary, earliest, latest, employer_summary]:
        features = features.merge(part, on="case_id", how="left", validate="one_to_one")

    for days in [30, 90, 180]:
        recent = x[x["_days_before_decision"].between(0, days)]
        window = recent.groupby("case_id", sort=False).agg(
            **{
                f"taxc_record_count_{days}d": ("num_group1", "size"),
                f"taxc_amount_sum_{days}d": (TAX_C_AMOUNT, "sum"),
                f"taxc_active_dates_{days}d": (TAX_C_DATE, "nunique"),
            }
        ).reset_index()
        features = features.merge(window, on="case_id", how="left", validate="one_to_one")
        window_cols = [
            f"taxc_record_count_{days}d",
            f"taxc_amount_sum_{days}d",
            f"taxc_active_dates_{days}d",
        ]
        features[window_cols] = features[window_cols].fillna(0)

    features = features.merge(base, on="case_id", how="left", validate="one_to_one")
    features["taxc_duplicate_record_count"] = (
        features["taxc_record_count"] - features["taxc_unique_record_count"]
    )
    features["taxc_duplicate_record_rate"] = (
        features["taxc_duplicate_record_count"] / features["taxc_record_count"]
    )
    features["taxc_date_span_days"] = (
        features["taxc_date_max"] - features["taxc_date_min"]
    ).dt.days
    features["taxc_latest_recency_days"] = (
        features["date_decision"] - features["taxc_date_max"]
    ).dt.days
    features["taxc_daily_amount_change"] = (
        features["taxc_latest_daily_amount"] - features["taxc_earliest_daily_amount"]
    )
    features["taxc_has_record"] = 1
    features["taxc_record_count_log1p"] = np.log1p(features["taxc_record_count"])
    features = features.drop(columns=["taxc_date_min", "taxc_date_max", "date_decision"])
    return features


### 第六步：执行聚合

生成每笔贷款申请一行的结果，并检查 `case_id` 是否唯一。`SAVE_RESULT=False` 时只查看结果，不写入文件。


In [ ]:
taxc_features = aggregate_tax_registry_c(tax_c, tax_c_base)

assert taxc_features["case_id"].is_unique
assert len(taxc_features) == tax_c["case_id"].nunique()

print("申请级结果：", taxc_features.shape)
display(taxc_features.head())

if SAVE_RESULT:
    taxc_features.to_parquet(TAX_C_OUTPUT, index=False)
    print("已保存：", TAX_C_OUTPUT)
